In [ ]:
from google.colab import files

uploaded = files.upload()

Saving honeypot_logs.logs.csv to honeypot_logs.logs.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv('honeypot_logs.logs.csv')

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np

# 🧪 Load your dataset
# df = pd.read_csv('your_dataset.csv')  # or however you're loading it

# 🧹 Clean and Feature Engineer
df['referrer'] = df['referrer'].fillna('')
df['userAgent'] = df['userAgent'].fillna('')
df['interactionType'] = df['interactionType'].fillna('unknown')

df['referrer_len'] = df['referrer'].apply(len)
df['userAgent_len'] = df['userAgent'].apply(len)
df['userAgent_bot_flag'] = df['userAgent'].str.contains('bot|crawl|spider|crawler|slurp', case=False, na=False).astype(int)

# 🏷️ Encode interactionType
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
interaction_encoded = encoder.fit_transform(df[['interactionType']])
interaction_df = pd.DataFrame(interaction_encoded, columns=encoder.get_feature_names_out(['interactionType']))

# 🧩 Combine all features
features = pd.concat([
    df[['referrer_len', 'userAgent_len', 'userAgent_bot_flag']],
    interaction_df
], axis=1)

# 🧪 Use botScore > 0.5 as Bot, else Human (or adjust as per your logic)
df['Label'] = df['botScore'].apply(lambda x: 1 if x >= 0.5 else 0)

# 🧠 Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(features, df['Label'], test_size=0.2, random_state=42)

# 🌲 Train Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 📊 Evaluate
y_pred = model.predict(X_test)

print("✅ Accuracy Score:", accuracy_score(y_test, y_pred))
print("\n✅ Classification Report:\n", classification_report(y_test, y_pred))
print("✅ Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 📈 Feature Importance
importances = pd.Series(model.feature_importances_, index=features.columns)
print("\n📊 Feature Importance:\n", importances.sort_values(ascending=False))


✅ Accuracy Score: 1.0

✅ Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        29
           1       1.00      1.00      1.00       211

    accuracy                           1.00       240
   macro avg       1.00      1.00      1.00       240
weighted avg       1.00      1.00      1.00       240

✅ Confusion Matrix:
 [[ 29   0]
 [  0 211]]

📊 Feature Importance:
 interactionType_hidden_random    0.850212
interactionType_hidden_link      0.032807
interactionType_fake_api         0.032561
interactionType_decoy_form       0.030730
interactionType_dynamic_trap     0.028058
interactionType_decoy_logout     0.022493
referrer_len                     0.002421
userAgent_len                    0.000408
userAgent_bot_flag               0.000309
dtype: float64


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

# ✅ Load your data
df = pd.read_csv("honeypot_logs.logs.csv")  # Replace with actual file or source

# ✅ Label Creation: Assume bots have a 'botScore' above a threshold (e.g., 0.5)
df['Label'] = df['botScore'].apply(lambda x: 1 if x >= 0.5 else 0)

# ✅ Feature Engineering
df['referrer'] = df['referrer'].fillna('')
df['referrer_len'] = df['referrer'].apply(len)

df['userAgent'] = df['userAgent'].fillna('')
df['userAgent_len'] = df['userAgent'].apply(len)

df['userAgent_bot_flag'] = df['userAgent'].str.contains('bot|crawl|spider|python', case=False, na=False).astype(int)

# ✅ Features WITHOUT interactionType to avoid data leakage
features = df[['referrer_len', 'userAgent_len', 'userAgent_bot_flag']]
labels = df['Label']

# ✅ Split data
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

# ✅ Train Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ✅ Predict
y_pred = model.predict(X_test)

# ✅ Evaluate
print("✅ Accuracy Score:", accuracy_score(y_test, y_pred))
print("\n✅ Classification Report:")
print(classification_report(y_test, y_pred))
print("✅ Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ✅ Feature Importance
importances = pd.Series(model.feature_importances_, index=features.columns).sort_values(ascending=False)
print("\n📊 Feature Importance:")
print(importances)

# ✅ Optional: View predictions
output_df = X_test.copy()
output_df['Actual'] = y_test
output_df['Predicted'] = y_pred
output_df['Predicted_Label'] = output_df['Predicted'].map({1: 'Bot', 0: 'Human'})
print("\n🔍 Sample Prediction Output:")
print(output_df.head())


✅ Accuracy Score: 0.875

✅ Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        29
           1       0.88      1.00      0.93       211

    accuracy                           0.88       240
   macro avg       0.44      0.50      0.47       240
weighted avg       0.77      0.88      0.82       240

✅ Confusion Matrix:
[[  0  29]
 [  1 210]]

📊 Feature Importance:
referrer_len          0.884039
userAgent_len         0.095880
userAgent_bot_flag    0.020081
dtype: float64

🔍 Sample Prediction Output:
      referrer_len  userAgent_len  userAgent_bot_flag  Actual  Predicted  \
1178            20             22                   1       1          1   
865             23             57                   0       1          1   
101             20             71                   1       1          1   
439             24             47                   1       1          1   
58              28             71       

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import resample
import warnings
warnings.filterwarnings("ignore")

# 🔹 Load your dataset
df = pd.read_csv('honeypot_logs.logs.csv')  # Replace with your dataset path

# 🔹 Extract relevant features
df['referrer_len'] = df['referrer'].fillna('').apply(len)
df['userAgent_len'] = df['userAgent'].fillna('').apply(len)
df['userAgent_bot_flag'] = df['userAgent'].str.contains('bot|crawl|spider|curl|wget', case=False, na=False).astype(int)

# 🔹 Extract Label (Assuming bots have score ≥ 0.5)
df['Label'] = df['botScore'].apply(lambda x: 1 if x >= 0.5 else 0)

# ✅ Optional: Upsample minority class (Human = 0)
df_majority = df[df.Label == 1]
df_minority = df[df.Label == 0]

df_minority_upsampled = resample(df_minority,
                                 replace=True,
                                 n_samples=len(df_majority),
                                 random_state=42)

df = pd.concat([df_majority, df_minority_upsampled])

# 🔹 Features and Labels
features = ['referrer_len', 'userAgent_len', 'userAgent_bot_flag']
X = df[features]
y = df['Label']

# 🔹 Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ✅ Train RandomForest with class_weight='balanced'
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

# 🔍 Predictions and Evaluation
y_pred = model.predict(X_test)

print("✅ Accuracy Score:", round(accuracy_score(y_test, y_pred), 4))
print("\n✅ Classification Report:\n", classification_report(y_test, y_pred))
print("\n✅ Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 📊 Feature Importance
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print("\n📊 Feature Importance:\n", importances)

# 🔍 Sample Predictions
sample = X_test.copy()
sample['Actual'] = y_test.values
sample['Predicted'] = y_pred
sample['Predicted_Label'] = sample['Predicted'].map({0: 'Human', 1: 'Bot'})
print("\n🔍 Sample Prediction Output:\n", sample.head())


✅ Accuracy Score: 0.6059

✅ Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.73      0.65       200
           1       0.65      0.48      0.55       206

    accuracy                           0.61       406
   macro avg       0.62      0.61      0.60       406
weighted avg       0.62      0.61      0.60       406


✅ Confusion Matrix:
 [[147  53]
 [107  99]]

📊 Feature Importance:
 referrer_len          0.891002
userAgent_len         0.093176
userAgent_bot_flag    0.015822
dtype: float64

🔍 Sample Prediction Output:
       referrer_len  userAgent_len  userAgent_bot_flag  Actual  Predicted  \
438             29             71                   1       0          0   
1164            18             11                   1       1          1   
1005            31             47                   1       1          0   
1058            23             57                   0       0          0   
1056            19             71 

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 📥 Load your data
# Assuming your dataset is in a DataFrame called `df`
# Make sure these columns exist in your df
# Columns: 'userAgent', 'referrer', 'interactionType', 'botScore', 'details.query.token', 'details.formData.honeypot_field', etc.

# 🧹 Fill NaNs
df['userAgent'] = df['userAgent'].fillna('')
df['referrer'] = df['referrer'].fillna('')
df['interactionType'] = df['interactionType'].fillna('unknown')

# 🧠 Feature Engineering
df['userAgent_len'] = df['userAgent'].str.len()
df['referrer_len'] = df['referrer'].str.len()

# UserAgent Category
df['userAgent_browser'] = df['userAgent'].str.extract(r'(Chrome|Firefox|Safari|Edge|Bot|Crawler|Scrapy|curl)', expand=False).fillna('Other')

# Referrer Domain
df['referrer_domain'] = df['referrer'].str.extract(r'https?://(?:www\.)?([^/]+)', expand=False).fillna('unknown')

# Token Present or Not
df['has_token'] = df['details.query.token'].notna().astype(int)

# Honeypot field filled or not
df['honeypot_filled'] = df['details.formData.honeypot_field'].notna().astype(int)

# Convert target label
df['label'] = df['botScore'].apply(lambda x: 1 if x > 0.5 else 0)

# One-hot encode interactionType, userAgent_browser, referrer_domain
interaction_ohe = pd.get_dummies(df['interactionType'], prefix='interactionType')
browser_ohe = pd.get_dummies(df['userAgent_browser'], prefix='browser')
domain_ohe = pd.get_dummies(df['referrer_domain'], prefix='domain')

# Combine features
features = pd.concat([
    df[['referrer_len', 'userAgent_len', 'userAgent_bot_flag', 'has_token', 'honeypot_filled']],
    interaction_ohe,
    browser_ohe,
    domain_ohe
], axis=1)

# 🎯 Train-test split
X_train, X_test, y_train, y_test = train_test_split(features, df['label'], test_size=0.2, random_state=42, stratify=df['label'])

# 🧠 Train the model
model = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# 🧪 Evaluate
y_pred = model.predict(X_test)

print("✅ Accuracy Score:", round(accuracy_score(y_test, y_pred), 4))
print("\n✅ Classification Report:\n", classification_report(y_test, y_pred))
print("✅ Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 📊 Feature Importance
importances = pd.Series(model.feature_importances_, index=features.columns)
importances = importances.sort_values(ascending=False)
print("\n📊 Top 10 Feature Importances:\n", importances.head(10))

# 🔍 Sample predictions
sample = X_test.copy()
sample['Actual'] = y_test
sample['Predicted'] = y_pred
sample['Predicted_Label'] = sample['Predicted'].map({0: 'Human', 1: 'Bot'})
print("\n🔍 Sample Prediction Output:\n", sample.head())


✅ Accuracy Score: 1.0

✅ Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       203
           1       1.00      1.00      1.00       203

    accuracy                           1.00       406
   macro avg       1.00      1.00      1.00       406
weighted avg       1.00      1.00      1.00       406

✅ Confusion Matrix:
 [[203   0]
 [  0 203]]

📊 Top 10 Feature Importances:
 interactionType_hidden_random    0.226311
has_token                        0.215881
honeypot_filled                  0.086248
interactionType_dynamic_trap     0.082806
interactionType_fake_api         0.065124
interactionType_hidden_link      0.056486
interactionType_decoy_form       0.052113
interactionType_decoy_logout     0.042150
referrer_len                     0.012689
domain_willis-morrison.org       0.005287
dtype: float64

🔍 Sample Prediction Output:
      referrer_len  userAgent_len  userAgent_bot_flag  has_token  \
89             2

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import resample
import warnings
warnings.filterwarnings("ignore")

# 🔹 Load Dataset
df = pd.read_csv('honeypot_logs.logs.csv')  # Update path if needed

# 🔹 Drop rows with missing target
df = df[df['botScore'].notna()]

# 🔹 Feature Engineering
df['referrer_len'] = df['referrer'].fillna('').apply(len)
df['userAgent_len'] = df['userAgent'].fillna('').apply(len)
df['userAgent_bot_flag'] = df['userAgent'].str.contains('bot|crawl|spider|curl|wget', case=False, na=False).astype(int)

# 🔹 Binary Label: 1 = Bot, 0 = Human
df['Label'] = df['botScore'].apply(lambda x: 1 if x >= 0.5 else 0)

# 🔹 Feature Set
features = ['referrer_len', 'userAgent_len', 'userAgent_bot_flag']
X = df[features]
y = df['Label']

# 🔹 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 🔹 Upsample minority class in training set only
train_df = pd.concat([X_train, y_train], axis=1)
majority = train_df[train_df.Label == 1]
minority = train_df[train_df.Label == 0]

minority_upsampled = resample(minority,
                              replace=True,
                              n_samples=len(majority),
                              random_state=42)

train_balanced = pd.concat([majority, minority_upsampled])
X_train_bal = train_balanced[features]
y_train_bal = train_balanced['Label']

# ✅ Random Forest Classifier
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced_subsample',  # balances using bootstrap sample
    max_depth=10  # helps prevent overfitting
)
model.fit(X_train_bal, y_train_bal)

# 🔍 Predictions
y_pred = model.predict(X_test)

# ✅ Evaluation
print("✅ Accuracy Score:", round(accuracy_score(y_test, y_pred), 4))
print("\n✅ Classification Report:\n", classification_report(y_test, y_pred, target_names=['Human', 'Bot']))
print("\n✅ Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 📊 Feature Importance
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print("\n📊 Feature Importance:\n", importances)

# 🔍 Sample Predictions
sample = X_test.copy()
sample['Actual'] = y_test.values
sample['Predicted'] = y_pred
sample['Predicted_Label'] = sample['Predicted'].map({0: 'Human', 1: 'Bot'})
print("\n🔍 Sample Prediction Output:\n", sample.head())


✅ Accuracy Score: 0.5667

✅ Classification Report:
               precision    recall  f1-score   support

       Human       0.20      0.59      0.30        37
         Bot       0.88      0.56      0.69       203

    accuracy                           0.57       240
   macro avg       0.54      0.58      0.49       240
weighted avg       0.78      0.57      0.63       240


✅ Confusion Matrix:
 [[ 22  15]
 [ 89 114]]

📊 Feature Importance:
 referrer_len          0.846103
userAgent_len         0.132468
userAgent_bot_flag    0.021430
dtype: float64

🔍 Sample Prediction Output:
       referrer_len  userAgent_len  userAgent_bot_flag  Actual  Predicted  \
553             24             71                   1       1          0   
1028            21             57                   0       1          1   
1066            23             71                   1       1          0   
794             25             71                   1       1          0   
953             32             22 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# Load your dataset
df = pd.read_csv('honeypot_logs.logs.csv')

# Feature Engineering
df['referrer_len'] = df['referrer'].fillna('').apply(len)
df['userAgent_len'] = df['userAgent'].fillna('').apply(len)
df['userAgent_bot_flag'] = df['userAgent'].str.contains('bot|crawl|spider|curl|wget', case=False, na=False).astype(int)
df['Label'] = df['botScore'].apply(lambda x: 1 if x >= 0.5 else 0)

# Split features and label
features = ['referrer_len', 'userAgent_len', 'userAgent_bot_flag']
X = df[features]
y = df['Label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Upsample minority class in training only
train_df = pd.concat([X_train, y_train], axis=1)
majority = train_df[train_df.Label == 1]
minority = train_df[train_df.Label == 0]

minority_upsampled = resample(minority,
                              replace=True,
                              n_samples=len(majority),
                              random_state=42)

train_balanced = pd.concat([majority, minority_upsampled])
X_train_bal = train_balanced[features]
y_train_bal = train_balanced['Label']

# Normalize (optional but helps with some models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Try Gradient Boosting or Logistic Regression
# model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5)
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train_bal)

# Predict
y_pred = model.predict(X_test_scaled)

# Evaluation
print("✅ Accuracy Score:", round(accuracy_score(y_test, y_pred), 4))
print("\n✅ Classification Report:\n", classification_report(y_test, y_pred, target_names=['Human', 'Bot']))
print("\n✅ Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Sample predictions
sample = X_test.copy()
sample['Actual'] = y_test.values
sample['Predicted'] = y_pred
sample['Predicted_Label'] = sample['Predicted'].map({0: 'Human', 1: 'Bot'})
print("\n🔍 Sample Prediction Output:\n", sample.head())


✅ Accuracy Score: 0.4792

✅ Classification Report:
               precision    recall  f1-score   support

       Human       0.10      0.30      0.15        37
         Bot       0.80      0.51      0.62       203

    accuracy                           0.48       240
   macro avg       0.45      0.40      0.39       240
weighted avg       0.69      0.48      0.55       240


✅ Confusion Matrix:
 [[ 11  26]
 [ 99 104]]

🔍 Sample Prediction Output:
       referrer_len  userAgent_len  userAgent_bot_flag  Actual  Predicted  \
553             24             71                   1       1          0   
1028            21             57                   0       1          1   
1066            23             71                   1       1          0   
794             25             71                   1       1          0   
953             32             22                   0       1          0   

     Predicted_Label  
553            Human  
1028             Bot  
1066           Human